[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-10b-production-records-feedback-bias-and-regression-alarms.ipynb)

# Advanced Discussion 10 (continued) — call records, feedback bias and regression alarms

A call-record schema, a simulated week of dashboard numbers, the bias in thumbs-up/down feedback, and how quickly a daily alarm notices a drop.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Simulation only; runs in seconds.

In [ ]:
%pip install -q numpy

In [ ]:
import numpy as np, math
from dataclasses import dataclass, field
rng = np.random.RandomState(0)

## 1. one record per model call: what to capture (everything needed to debug, cost and audit)

In [ ]:
@dataclass
class LLMCallRecord:
    request_id: str; task_id: str; step: int; model: str; prompt_version: str
    input_tokens: int; cached_tokens: int; output_tokens: int; latency_s: float; cost: float
    tool_calls: list = field(default_factory=list); error: str = ""; user_feedback: int = 0        # +1 thumbs up, -1 thumbs down, 0 none
IN, OUT, READ = 3.00, 15.00, 0.10
def price(inp, cached, out): return ((inp - cached) * IN + cached * IN * READ + out * OUT) / 1e6
print("record fields:", [f for f in LLMCallRecord.__dataclass_fields__])

## 2. simulate a week of traffic, and what the dashboard should show

In [ ]:
N_TASKS = 4000
tasks = []
for i in range(N_TASKS):
    steps = 1 + rng.poisson(3.0); latency = rng.lognormal(math.log(1.2), 0.5, steps).sum(); tokens_in = int(rng.lognormal(math.log(9000), 0.6)); out = 120 * steps
    cost = price(tokens_in, int(0.7 * tokens_in), out); success = rng.rand() < 0.86
    tasks.append((steps, latency, cost, success))
steps, lat, cost, succ = map(np.array, zip(*tasks))
print("\nweek of traffic: %d tasks" % N_TASKS)
print("  success rate %.3f | latency p50 %.1f s, p95 %.1f s, p99 %.1f s | cost p50 $%.4f, p95 $%.4f, p99 $%.4f, total $%.0f" % (
    succ.mean(), *np.percentile(lat, [50, 95, 99]), *np.percentile(cost, [50, 95, 99]), cost.sum()))
print("  steps per task: mean %.1f, p95 %d" % (steps.mean(), np.percentile(steps, 95)))

## 3. user feedback is a biased sample

In [ ]:
# unhappy users are more likely to click; suppose 86% are satisfied, only 2% of satisfied users click 'thumbs up', and 12% of dissatisfied users click 'thumbs down'
satisfied = succ
click = np.where(satisfied, rng.rand(N_TASKS) < 0.02, rng.rand(N_TASKS) < 0.12); up = click & satisfied; down = click & ~satisfied
print("\nfeedback: %d thumbs up, %d thumbs down (%.1f%% of tasks gave feedback)" % (up.sum(), down.sum(), 100 * click.mean()))
print("  naive 'thumbs-up share' among those who clicked: %.2f   | true satisfaction: %.2f" % (up.sum() / click.sum(), satisfied.mean()))
# an audit: read a random sample of 200 tasks and grade them
aud = rng.choice(N_TASKS, 200, replace=False); p = satisfied[aud].mean(); half = 1.96 * math.sqrt(p * (1 - p) / 200)
print("  random audit of 200 graded tasks: %.2f +/- %.2f (95%%)" % (p, half))
print("  but thumbs-down counts are still useful as a triage queue: %d tasks flagged, of which %.0f%% really failed" % (down.sum(), 100 * (~satisfied[down]).mean()))
print("  failures never flagged: %d of %d (%.0f%%)" % ((~satisfied & ~click).sum(), (~satisfied).sum(), 100 * (~satisfied & ~click).sum() / (~satisfied).sum()))

## 4. spotting a regression from day-by-day dashboards: is a drop real or noise?

In [ ]:
days = 14; per_day = 300; base = 0.86
def daily(rate_after, change_day=8, seed=3):
    r = np.random.RandomState(seed); out = []
    for d in range(days): p = base if d < change_day else rate_after; out.append(r.binomial(per_day, p) / per_day)
    return np.array(out)
d = daily(0.80)
print("\ndaily success rate with %d tasks/day (true rate 0.86, drops to 0.80 on day 9):" % per_day)
print("  " + " ".join("%.2f" % x for x in d))
half = 1.96 * math.sqrt(base * (1 - base) / per_day); print("  day-to-day noise: 95%% of days should fall within +/- %.3f of the true rate" % half)
def detect_day(rate_after, thresh=3, runs=2000):
    """Alert when the daily rate is below base - 2 sigma on `thresh` consecutive days."""
    lim = base - 2 * math.sqrt(base * (1 - base) / per_day); found = []; false = 0
    for s in range(runs):
        x = daily(rate_after, seed=s); streak = 0; hit = None
        for i, v in enumerate(x):
            streak = streak + 1 if v < lim else 0
            if streak >= thresh and hit is None: hit = i
        found.append(hit)
    hits = [h for h in found if h is not None and h >= 8]; early = [h for h in found if h is not None and h < 8]
    return len(hits) / runs, (np.mean(hits) - 8 + 1) if hits else float("nan"), len(early) / runs
print("alert rule: below the 2-sigma line 3 days running. detection rate, mean days to alert, false alarm rate before the change:")
for after in (0.84, 0.80, 0.74):
    dr, md, fa = detect_day(after); print("  true drop to %.2f: detected %.2f, %.1f days after the change, false alarms %.3f" % (after, dr, md, fa))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.